# 08 · Figures

This notebook draws every figure of the paper: Figs. 1–5 and Supplementary Figs. 1–3. Each figure has its own section with the legend of the paper, the code and the figure.

| | |
|---|---|
| Reads | `SUMMARY/outcomes_summary.csv` and `SUMMARY/outcomes_by_seed.csv` (notebook 05) for Figs. 2–5 and Supplementary Fig. 2; for the ECG example of Fig. 1a also `PTBXL_ARRAY`, `PTBXL_SETS` and the logits files of two ECGFounder models (notebooks 01, 02, 04); `SET_COUNTS` (notebook 02) for Supplementary Fig. 1; `TIMING/adaptation_time.csv` and `TIMING/reading_time.csv` (notebook 06) for Supplementary Fig. 3 |
| Writes | `FIGURES/Fig1` … `Fig5` and `FIGURES/SuppFig1` … `SuppFig3`, each as `.png` (300 dots per inch) and `.pdf` (vector graphics, text kept as text) |
| Needs | no GPU. Less than 1 min |

## Outline

1. **Plotting style**: font, sizes, colours and the helper functions that all figures share.
2. **Data**: the summary tables of notebook 05 and the names of the configurations.
3. **Fig. 1** to **Fig. 5**.
4. **Supplementary Fig. 1** to **Supplementary Fig. 3**.
5. **Files written**.

Every result that is drawn or printed in a figure comes from the files named above. Set by hand are only the layout (positions in millimetres), the wording of the labels,
and the axis limits of a few panels, which are fixed to those of the paper (see **Axis limits** below).

In [ ]:
%run ./00_setup_and_settings.ipynb

In [ ]:
import json

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Patch
from matplotlib.ticker import MultipleLocator

## Plotting style

The figures follow the usual requirements of journals for a figure of two columns: a width of 180 mm, text of 5 to 7 points, panel letters of 8 points in bold lower case, and lines of at least 0.4 points.
All positions are given in millimetres from the top left corner of the figure, so that the size of every panel is fixed and does not depend on the length of a label.

### Font

The figures of the paper use **Arial**. If Arial is not installed, the first font of the list below that is found is used (they have the same or nearly the same widths as Arial),
and otherwise the default font of matplotlib (DejaVu Sans). The notebook then still runs and writes all figures, but DejaVu Sans is wider than Arial:
some long labels overlap their neighbours or are cut at the edge of the figure. Install Arial or one of the other fonts of the list to obtain the figures of the paper.

In [ ]:
FONT = None
for _name in ("Arial", "Helvetica", "Liberation Sans", "Arimo"):
    try:
        font_manager.findfont(_name, fallback_to_default=False)      # raises an error when the font is not installed
        FONT = _name
        break
    except Exception:
        continue
print("font of the figures:", FONT if FONT else f"{font_manager.FontProperties().get_name()} (default of matplotlib; Arial was not found)")

### Sizes and colours

The settings start from the defaults of matplotlib, so that a local configuration file does not change the figures.

Colours carry meaning and are the same in all figures:

- the three remedies: training (distillation) blue, reading (mixed reading) green, head (penalized head) orange; the baseline is grey;
- the three foundation models: ECGFounder black with circles, HuBERT-ECG violet with squares, ECG-FM ochre with triangles. Each model has its own marker, so that the models can be told apart without colour.

In [ ]:
MM = 1 / 25.4                 # inches per millimetre
WIDTH_MM = 180.0              # width of every figure
PNG_DPI = 300                 # resolution of the .png files (2,125 pixels for 180 mm)

C_TRAIN, C_READ, C_HEAD, C_BASE = "#2a78d6", "#1baf7a", "#eb6834", "#898781"      # training, reading, head, baseline
INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e6e5df", "#6b6a65"   # text, secondary text, dashed outlines, grid lines, axes
MODEL_COLOR = {"ecgfounder": "#0b0b0b", "hubert": "#7a5cc7", "ecgfm": "#b8860b"}
MODEL_MARKER = {"ecgfounder": "o", "hubert": "s", "ecgfm": "^"}

STYLE = {
    "font.size": 6, "axes.labelsize": 6, "axes.titlesize": 6.5, "xtick.labelsize": 5.5, "ytick.labelsize": 5.5, "legend.fontsize": 5.5,
    "axes.linewidth": 0.5, "axes.edgecolor": AXIS, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": AXIS, "ytick.color": AXIS, "xtick.labelcolor": INK, "ytick.labelcolor": INK,
    "xtick.major.width": 0.5, "ytick.major.width": 0.5, "xtick.minor.width": 0.4, "ytick.minor.width": 0.4,
    "xtick.major.size": 2, "ytick.major.size": 2, "xtick.major.pad": 1.5, "ytick.major.pad": 1.5,
    "axes.spines.top": False, "axes.spines.right": False, "axes.titlepad": 3, "axes.labelpad": 2, "axes.titleweight": "bold",
    "lines.linewidth": 0.9, "lines.markersize": 3, "lines.markeredgewidth": 0.6, "errorbar.capsize": 0,
    "legend.frameon": False, "legend.handlelength": 1.3, "legend.handletextpad": 0.4, "legend.borderaxespad": 0.2,
    "legend.borderpad": 0.2, "legend.labelspacing": 0.25, "legend.columnspacing": 0.8,
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",       # fonts are embedded as text, so that the labels can be edited in the .pdf files
    "figure.dpi": 150, "figure.facecolor": "white",
}
if FONT:
    STYLE["font.family"] = FONT
matplotlib.rcdefaults()
plt.rcParams.update(STYLE)

### Helper functions

- `figure`, `axes_mm`, `panel_letter`, `text_mm`: a figure of 180 mm width and its parts, placed in millimetres from the top left corner;
- `tint`: a colour mixed with white, for backgrounds and boxes;
- `grid`: light grid lines behind the data;
- `save_and_show`: writes the `.pdf` and the `.png` file to `FIGURES` and shows the `.png` file below the cell, so that what is seen in the notebook is the file that was written;
- `spread`: moves labels apart that would overlap;
- `rounded`: rounds a number half away from zero, as the tables of the paper do (the built-in rounding of Python rounds 0.5 to the even neighbour);
- `axis_limits`: see **Axis limits**.

In [ ]:
def figure(height_mm, width_mm=WIDTH_MM):
    """A new figure of the given size in millimetres."""
    return plt.figure(figsize=(width_mm * MM, height_mm * MM))


def axes_mm(fig, x, y, w, h, **kw):
    """Axes placed in millimetres: x, y = top left corner of the plotting area measured from the top left corner of the figure; w, h = its size."""
    W, H = fig.get_size_inches() / MM
    return fig.add_axes([x / W, 1 - (y + h) / H, w / W, h / H], **kw)


def text_mm(fig, x, y, s, **kw):
    """Text at a position in millimetres from the top left corner of the figure."""
    W, H = fig.get_size_inches() / MM
    return fig.text(x / W, 1 - y / H, s, **kw)


def panel_letter(fig, x, y, s):
    """Panel letter: 8 points, bold, lower case."""
    return text_mm(fig, x, y, s, fontsize=8, fontweight="bold", va="top", ha="left")


def tint(color, a):
    """The colour mixed with white (a = share of the colour)."""
    c = np.array(matplotlib.colors.to_rgb(color))
    return tuple(1 - a * (1 - c))


def grid(ax, axis):
    """Light grid lines along one axis ("x" or "y"), behind the data."""
    ax.grid(axis=axis, color=GRID, lw=0.4, zorder=0)
    ax.set_axisbelow(True)


def save_and_show(fig, name):
    """Write FIGURES/name.pdf and FIGURES/name.png, close the figure and show the .png file."""
    fig.savefig(FIGURES / f"{name}.pdf")
    fig.savefig(FIGURES / f"{name}.png", dpi=PNG_DPI)
    plt.close(fig)
    display(Image(filename=str(FIGURES / f"{name}.png"), width=1000))


def spread(values, gap):
    """Positions for labels next to the values: neighbours are moved apart until they are at least `gap` from each other."""
    order = np.argsort(values); pos = np.array(values, float)[order]
    for _ in range(200):
        moved = False
        for i in range(1, len(pos)):
            if pos[i] - pos[i - 1] < gap - 1e-9:
                mid = (pos[i] + pos[i - 1]) / 2; pos[i - 1] = mid - gap / 2; pos[i] = mid + gap / 2; moved = True
        if not moved:
            break
    out = np.empty(len(pos)); out[order] = pos
    return out


def rounded(v, digits=1):
    """Round half away from zero (with a small tolerance for numbers that are stored just below a half)."""
    v = float(v); sign = -1.0 if v < 0 else 1.0
    return sign * np.floor(abs(v) * 10 ** digits + 0.5 + 2e-5) / 10 ** digits

### Axis limits

In the forest plots (Figs. 3 and 4) and in Supplementary Fig. 2 the axis has to hold the confidence intervals. Limits calculated from the confidence limits would change a little with every new set of bootstrap resamples,
and all symbols would move with them. The limits of these panels are therefore fixed to those of the figures in the paper (`PAPER_LIMITS`).
If a value lies outside the fixed limits, which can happen with other models or other data, `axis_limits` returns limits with a margin of 8% around the values instead.

In [ ]:
PAPER_LIMITS = {
    ("Fig3", "adjacent_ptbxl"): (-13.51, 1.001),                # difference in decision change, percentage points
    ("Fig3", "auroc_ptbxl"): (-0.006897, 0.009636),            # difference in AUROC
    ("Fig4", "auroc"): (-0.02365, 0.08287),                     # differences in the five AUROCs
    ("Fig4", "direction"): (-4.228, 6.740),                    # differences in direction agreement, percentage points
    ("SuppFig2", "adjacent_ptbxl"): (9.076, 26.29),             # decision change, %
    ("SuppFig2", "auroc_ptbxl"): (-0.01509, 0.005417),          # difference in AUROC
}


def axis_limits(key, values, margin=0.08):
    """Limits of the paper for this panel; if a value lies outside them, limits with a margin around all values."""
    lo, hi = float(np.min(values)), float(np.max(values))
    paper = PAPER_LIMITS[key]
    if paper[0] <= lo and hi <= paper[1]:
        return paper
    pad = margin * (hi - lo)
    return lo - pad, hi + pad

## Data

Notebook 05 wrote two tables that hold everything the figures show (their columns are documented in the last section of that notebook):

- `outcomes_summary.csv`: one row for each configuration (training, head, reading) of each foundation model. For every outcome it has the value (mean of the three seeds) with its 95% bootstrap confidence interval
  (`{outcome}_value`, `_value_lo`, `_value_hi`), the paired difference from the baseline of the same foundation model with its interval (`{outcome}_diff`, `_diff_lo`, `_diff_hi`),
  whether this interval excludes zero (`{outcome}_sig`), and for decision change the relative reduction from the baseline (`{outcome}_relative_reduction`).
- `outcomes_by_seed.csv`: one row for each configuration and seed, with the decision thresholds of that predictor. Only Fig. 1a uses it.

Proportions are in percent and their differences in percentage points. AUROCs are on the scale 0 to 1.

In [ ]:
for _f in ("outcomes_summary.csv", "outcomes_by_seed.csv"):
    if not (SUMMARY / _f).exists():
        raise FileNotFoundError(f"{_f} is missing in SUMMARY: run notebook 05 first")
SUM = pd.read_csv(SUMMARY / "outcomes_summary.csv")
BY_SEED = pd.read_csv(SUMMARY / "outcomes_by_seed.csv")
print(f"{len(SUM)} configurations, {SUM.shape[1]} columns | {len(BY_SEED)} rows by seed")

### Names of the configurations

A configuration is a combination of training, head and reading (names as in notebook 05). The **baseline** is the model fine-tuned on labels only, with the head trained with the network and a single read of the segment.
The **three remedies** are the distilled model (`student`), the penalized head with the weight of the final setting, and mixed reading.
The remedies are added one at a time in Fig. 2 (`STEPS`), and each is applied alone in Fig. 2c (`ALONE`).

In [ ]:
HEAD = f"pen{PENALTY_WEIGHT}"                                      # penalized head of the final setting: "pen300"
MIXED = f"mixed{1 - MIX_SINGLE:g}"                                 # mixed reading of the final setting: "mixed0.9"
BASELINE = ("baseline", "model", "single")                         # training, head, reading
REMEDIES = ("student", HEAD, MIXED)
STEPS = [("Baseline", BASELINE), ("+ Distillation", ("student", "model", "single")), ("+ Mixed reading", ("student", "model", MIXED)), ("+ Penalized head", REMEDIES)]
ALONE = [("Distillation alone", ("student", "model", "single"), C_TRAIN), ("Mixed reading alone", ("baseline", "model", MIXED), C_READ),
         ("Penalized head alone", ("baseline", HEAD, "single"), C_HEAD), ("All three", REMEDIES, INK)]

OUT_ADJACENT = ["adjacent_ptbxl", "adjacent_mimic", "adjacent_code15", "adjacent_echonext"]      # decision change between adjacent segments, four datasets
OUT_BETWEEN = ["between_same_report", "between_all"]                                             # decision change between recordings; pairs with identical machine reports first
OUT_AUROC = ["auroc_ptbxl", "auroc_mimic", "auroc_code15_cd", "auroc_code15_normal", "auroc_echonext"]
OUT_DIRECTION = ["direction_cd", "direction_normal"]
OUTCOME_NAME = {"adjacent_ptbxl": "PTB-XL", "adjacent_mimic": "MIMIC-IV-ECG", "adjacent_code15": "CODE-15%", "adjacent_echonext": "EchoNext",
                "between_same_report": "Identical machine report", "between_all": "All pairs",
                "auroc_ptbxl": "PTB-XL, five diagnostic classes", "auroc_mimic": "MIMIC-IV-ECG, five diagnostic classes", "auroc_code15_cd": "CODE-15%, conduction disturbance",
                "auroc_code15_normal": "CODE-15%, normal ECG", "auroc_echonext": "EchoNext, wall thickness",
                "direction_cd": "CODE-15%, conduction disturbance", "direction_normal": "CODE-15%, normal ECG"}


def config(model, training, head, reading):
    """The row of one configuration in the summary table."""
    q = SUM[(SUM["model"] == model) & (SUM["training"] == training) & (SUM["head"] == head) & (SUM["reading"] == reading)]
    assert len(q) == 1, f"{len(q)} rows for {model}, {training}, {head}, {reading} in outcomes_summary.csv"
    return q.iloc[0]


def value_ci(row, outcome):
    """Value of an outcome with its 95% CI: (value, lower limit, upper limit)."""
    return tuple(float(row[f"{outcome}_value{s}"]) for s in ("", "_lo", "_hi"))


def diff_ci(row, outcome):
    """Paired difference from the baseline with its 95% CI, and whether the interval excludes zero: (difference, lower limit, upper limit, True or False)."""
    return (*(float(row[f"{outcome}_diff{s}"]) for s in ("", "_lo", "_hi")), str(row[f"{outcome}_sig"]) == "True")

## Fig. 1

**Fig. 1 | Decision changes between adjacent segments of a recording and three remedies based on within-recording consistency.** **a**, Example from the PTB-XL test fold. The 10-s recording is divided into two adjacent 5-s segments, and each is read as one ECG. Right, probability of myocardial infarction given by ECGFounder for the two segments at baseline and with the three remedies, with the decision thresholds (vertical lines). At baseline the two probabilities lie on either side of the threshold, so the same heart receives two different decisions. The remedies aim to bring the two probabilities to the same side of the threshold, as in this example. **b**, The three within-recording remedies. Each uses overlapping windows of the segment being read as replicate readings. Training: a student that reads the whole segment once is trained to match the mean output of a teacher over seven 2.5-s windows. Reading: the probability is a weighted average of one read of the segment (weight 0.1) and the mean over 25 overlapping 2.5-s windows (weight 0.9). Head: a linear head is refitted with a penalty on the directions in which the embeddings of two 4.5-s views of the same segment, 0.5 s apart, differ. **c**, Design of this study: outcomes assessed in each dataset (filled circles). Repeatability was the primary outcome, and discrimination and the response to documented change were safety checks.

### Data of the example (panel a)

The example is one recording of the PTB-XL test fold (`ecg_id` 19867, labelled as a normal ECG) and the output for myocardial infarction of two ECGFounder models of **one training seed** (seed 0):

- baseline: the baseline model, probability of a single read of each segment;
- three remedies: the distilled model with the penalized head and mixed reading, that is `MIX_SINGLE` × probability of the single read + (1 − `MIX_SINGLE`) × mean probability of the 25 windows.

Nothing is averaged over seeds here, because a decision is made by one model. The thresholds are those of the same two predictors (`threshold_MI` in `outcomes_by_seed.csv`).
The cell prints the numbers that the figure shows.
The row of the recording in a logits file is found with the arrays `set` and `record` that every logits file contains.
Three of the twelve leads are drawn (II, V2 and V5).

In [ ]:
EXAMPLE_ECG_ID = 19867        # recording of the PTB-XL test fold
EXAMPLE_SEED = 0              # training seed of the two models that are shown
EXAMPLE_CLASS = "MI"          # myocardial infarction
EXAMPLE_LEADS = ["II", "V2", "V5"]


def example_probabilities(tag, head, mixed):
    """Probability of EXAMPLE_CLASS for the two segments of the example recording, from the logits file of one fine-tuned model."""
    z = np.load(LOGITS / f"{tag}.npz")
    rows = np.flatnonzero((z["set"] == "ptbxl_test") & (z["record"] == EXAMPLE_ECG_ID))
    assert len(rows) == 1, f"recording {EXAMPLE_ECG_ID} was not found once in the PTB-XL test rows of {tag}.npz"
    logistic = lambda logit: 1.0 / (1.0 + np.exp(-logit.astype(np.float64)))
    p = logistic(z[f"{head}_single"][rows[0]])                                     # (2 segments, 5 classes)
    if mixed:
        p = MIX_SINGLE * p + (1 - MIX_SINGLE) * logistic(z[f"{head}_windows"][rows[0]]).mean(1)      # probabilities are averaged over the 25 windows, not logits
    return p[:, CLASSES.index(EXAMPLE_CLASS)]


def example_threshold(training, head, reading, seed):
    q = BY_SEED[(BY_SEED["model"] == "ecgfounder") & (BY_SEED["training"] == training) & (BY_SEED["head"] == head) & (BY_SEED["reading"] == reading) & (BY_SEED["seed"] == seed)]
    assert len(q) == 1
    return float(q[f"threshold_{EXAMPLE_CLASS}"].iloc[0])


def example(seed):
    """For the two models of one seed: (probabilities of the two segments, threshold) at baseline and with the three remedies."""
    return {"baseline": (example_probabilities(model_tag("ecgfounder", "baseline", seed), "model", mixed=False), example_threshold(*BASELINE, seed)),
            "remedies": (example_probabilities(model_tag("ecgfounder", "student", seed), HEAD, mixed=True), example_threshold(*REMEDIES, seed))}


_sets = pd.read_csv(PTBXL_SETS)
_row = int(_sets.loc[_sets["ecg_id"] == EXAMPLE_ECG_ID, "row"].iloc[0])
EXAMPLE_ECG = np.asarray(np.load(PTBXL_ARRAY, mmap_mode="r")[_row]).astype(float)        # (12 leads, 5000 samples), mV
EXAMPLE = example(EXAMPLE_SEED)
print(f"recording {EXAMPLE_ECG_ID}, models of seed {EXAMPLE_SEED}, probability of {CLASS_NAMES[EXAMPLE_CLASS].lower()}")
for _k, (_p, _t) in EXAMPLE.items():
    print(f"   {_k:9s} segment 1 {_p[0]:.3f}, segment 2 {_p[1]:.3f} | threshold {_t:.3f} | decisions {'differ' if (_p[0] >= _t) != (_p[1] >= _t) else 'agree'}")

### Panel a: the example

Left, the three leads with the two segments shaded. Right, the two probabilities of each predictor on one line, with its threshold.
Whether the two decisions agree is read from the data: a red cross is drawn when the two probabilities lie on different sides of the threshold, and a green tick when they lie on the same side.

In [ ]:
def fig1_panel_a(fig):
    shade1, shade2 = tint(C_TRAIN, 0.10), tint(C_HEAD, 0.10); BAD, GOOD = "#b3261e", "#1a7f37"
    panel_letter(fig, 2, 3, "a")
    # ECG: 10 s, three leads, one below the other
    ax = axes_mm(fig, 9, 10, 106, 33); t = np.arange(N10) / FS
    for name, offset in zip(EXAMPLE_LEADS, (0, -2.3, -4.6)):
        ax.plot(t, EXAMPLE_ECG[LEADS.index(name)] + offset, lw=0.45, color=INK); ax.text(-0.12, offset, name, ha="right", va="center", fontsize=5.5)
    ax.axvspan(0, 5, color=shade1, lw=0); ax.axvspan(5, 10, color=shade2, lw=0)
    ax.set_xlim(0, 10); ax.set_ylim(-6.9, 1.6); ax.axis("off")
    ax.plot([0.15, 1.15], [-6.0, -6.0], color=INK, lw=0.8); ax.text(0.65, -6.12, "1 s", ha="center", va="top", fontsize=5)          # scale bars
    ax.plot([0.15, 0.15], [-6.0, -5.0], color=INK, lw=0.8); ax.text(0.22, -5.5, "1 mV", ha="left", va="center", fontsize=5)
    text_mm(fig, 9 + 26.5, 8.6, "Segment 1 (0-5 s), read as one ECG", fontsize=6, ha="center", va="bottom", color=C_TRAIN, fontweight="bold")
    text_mm(fig, 9 + 79.5, 8.6, "Segment 2 (5-10 s), read as one ECG", fontsize=6, ha="center", va="bottom", color=C_HEAD, fontweight="bold")
    # probabilities of the two segments and the threshold, for the baseline and for the three remedies
    ax = axes_mm(fig, 137, 15, 38, 23); xmax = 0.80
    for y, (p, threshold) in enumerate((EXAMPLE["baseline"], EXAMPLE["remedies"])):
        same = (p[0] >= threshold) == (p[1] >= threshold)
        col, verdict = (GOOD, "same\ndecision") if same else (BAD, "different\ndecisions")
        ax.axhspan(y - 0.46, y + 0.46, color=tint(col, 0.07), lw=0, zorder=0)
        ax.plot([p[0], p[1]], [y, y], color=tint(INK, 0.25), lw=1.6, solid_capstyle="round", zorder=1)
        ax.scatter([p[0]], [y], s=22, color=C_TRAIN, zorder=3); ax.scatter([p[1]], [y], s=22, color=C_HEAD, zorder=3)
        ax.plot([threshold, threshold], [y - 0.36, y + 0.36], color=INK, lw=0.9, zorder=2)
        ax.text(xmax - 0.015, y, verdict, fontsize=5.6, va="center", ha="right", color=col, fontweight="bold", linespacing=0.95)
        xm = xmax - 0.275                                             # cross or tick, drawn with lines
        if same:
            ax.plot([xm - 0.02, xm - 0.006, xm + 0.022], [y + 0.0, y + 0.13, y - 0.14], color=col, lw=1.1, zorder=4)
        else:
            ax.plot([xm - 0.018, xm + 0.018], [y - 0.13, y + 0.13], color=col, lw=1.1, zorder=4); ax.plot([xm - 0.018, xm + 0.018], [y + 0.13, y - 0.13], color=col, lw=1.1, zorder=4)
    ax.text(EXAMPLE["baseline"][1], -0.50, "decision threshold", fontsize=5, ha="center", va="bottom", color=INK2)
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Baseline", "With the three\nremedies"]); ax.set_ylim(1.62, -0.85); ax.set_xlim(0, xmax); ax.set_xticks([0, 0.2, 0.4, 0.6])
    ax.set_xlabel(f"Probability of {CLASS_NAMES[EXAMPLE_CLASS].lower()}"); grid(ax, "x")
    text_mm(fig, 126, 8.6, "The same heart should receive the same decision", fontsize=6, ha="left", va="bottom", fontweight="bold")
    ax.scatter([], [], s=12, color=C_TRAIN, label="Segment 1"); ax.scatter([], [], s=12, color=C_HEAD, label="Segment 2")
    ax.legend(loc="upper left", bbox_to_anchor=(-0.02, -0.30), ncol=2, handletextpad=0.1, columnspacing=1.0, borderaxespad=0.0)

### Panel b: the three remedies

A diagram without data, drawn in millimetres on one canvas. The bars are drawn to scale: a 2.5-s window is half as long as the 5-s segment, a 4.5-s view is 0.9 of it, and the two views are 0.1 of it (0.5 s) apart.
Of each series of windows the first, the second and the last are drawn, with dots for the windows in between.
The numbers in the labels are the constants of notebook 00 (`N_WIN_TEACHER`, `N_WIN_READ`, `MIX_SINGLE`, `WIN`, `VIEW`, `VIEW_SHIFT`).

In [ ]:
def fig1_panel_b(fig):
    NUMBER = {7: "seven"}
    win_s, view_s, shift_s, seg_s = WIN / FS, VIEW / FS, VIEW_SHIFT / FS, SEG / FS
    panel_letter(fig, 2, 51, "b")
    text_mm(fig, 8, 53.2, "Three within-recording remedies: overlapping windows of the segment serve as replicate readings", fontsize=6.3, fontweight="bold", va="center", ha="left")
    cv = axes_mm(fig, 4, 57, 112, 70); cv.set_xlim(0, 112); cv.set_ylim(70, 0); cv.axis("off")
    seg_w = 30.0                                                  # drawn width of one 5-s segment (mm)

    def box(x, y, w, h, fc, ec="none", lw=0.5, r=0.6):
        cv.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=f"round,pad=0,rounding_size={r}", fc=fc, ec=ec, lw=lw, zorder=2))

    def arrow(x0, y0, x1, y1, col=INK2, lw=0.6):
        cv.add_patch(FancyArrowPatch((x0, y0), (x1, y1), arrowstyle="-|>", mutation_scale=4.5, color=col, lw=lw, shrinkA=0, shrinkB=0, zorder=4))

    def txt(x, y, s, **kw):
        kw.setdefault("fontsize", 5.6); kw.setdefault("va", "center"); kw.setdefault("ha", "left"); cv.text(x, y, s, **kw)

    def windows(x, y, col, h=1.25, gap=0.5):
        """First, second and last window of a series, with dots for the windows that are not drawn. Returns the y position below the series."""
        ww = seg_w * win_s / seg_s; step = (seg_w - ww) * 0.14
        box(x, y, ww, h, col, r=0.3); box(x + step, y + (h + gap), ww, h, col, r=0.3)
        txt(x + seg_w / 2, y + 2 * (h + gap) + 0.45, ". . .", fontsize=6.5, ha="center", color=INK2, fontweight="bold")
        box(x + seg_w - ww, y + 3 * (h + gap) + 0.2, ww, h, col, r=0.3)
        return y + 4 * (h + gap) + 0.2

    rows = [1, 25, 48]
    for y0, col, title, sub in ((rows[0], C_TRAIN, "Training: distillation from a short-window teacher", "the student reads the whole segment once and learns to match the teacher's average over short windows"),
                                (rows[1], C_READ, "Reading: mixed reading", f"the output is a weighted average of one read of the segment and the mean over {N_WIN_READ} overlapping {win_s:g}-s windows"),
                                (rows[2], C_HEAD, "Head: penalized linear head", "the head is refitted with a penalty on directions in which two shifted views of the same segment differ")):
        box(0, y0, 1.4, 19.5, col, r=0.3); txt(3.2, y0 + 1.6, title, fontsize=6.3, fontweight="bold", color=col); txt(3.2, y0 + 4.6, sub, fontsize=5.2, color=INK2)
    # training: the student reads the segment once, the fixed teacher reads the windows, and the two outputs are matched
    y = rows[0] + 9.0
    box(4, y + 0.2, seg_w, 2.6, tint(INK, 0.16)); txt(4 + seg_w / 2, y - 1.3, f"one {seg_s:g}-s segment", fontsize=5, ha="center", color=INK2)
    yb = windows(4, y + 4.2, tint(C_TRAIN, 0.45)); txt(4 + seg_w / 2, yb + 0.9, f"{NUMBER.get(N_WIN_TEACHER, N_WIN_TEACHER)} {win_s:g}-s windows", fontsize=5, ha="center", color=INK2)
    box(41, y - 1.2, 21, 5.4, "white", ec=C_TRAIN, lw=0.7); txt(51.5, y + 1.5, "Student model", ha="center", fontsize=5.6)
    box(41, y + 5.4, 21, 5.4, tint(C_TRAIN, 0.16), ec=C_TRAIN, lw=0.7); txt(51.5, y + 8.1, "Teacher (fixed)", ha="center", fontsize=5.6)
    arrow(34.6, y + 1.5, 40.6, y + 1.5); arrow(34.6, y + 8.1, 40.6, y + 8.1)
    arrow(62.4, y + 1.5, 68, y + 1.5); arrow(62.4, y + 8.1, 68, y + 8.1)
    txt(69, y + 1.5, "output", fontsize=5.6); txt(69, y + 8.1, "mean output", fontsize=5.6)
    cv.plot([83.2, 84.8, 84.8, 83.2], [y + 1.5, y + 1.5, y + 8.1, y + 8.1], color=C_TRAIN, lw=0.8, zorder=4, solid_joinstyle="miter")    # bracket that joins the two outputs that are matched
    cv.plot([84.8, 86.4], [y + 4.8, y + 4.8], color=C_TRAIN, lw=0.8, zorder=4)
    txt(87.2, y + 4.8, "matched during training,\non labelled and unlabelled ECGs", fontsize=5.2, color=C_TRAIN)
    # reading: weighted average of one read and of the mean over the windows
    y = rows[1] + 8.4
    box(4, y, seg_w, 2.6, tint(INK, 0.16)); txt(36.5, y + 1.3, "one read", fontsize=5.6); txt(52, y + 1.3, f"x {MIX_SINGLE:g}", fontsize=5.8, fontweight="bold", color=C_READ)
    yb = windows(4, y + 3.9, tint(C_READ, 0.45)); txt(36.5, y + 7.2, f"mean of {N_WIN_READ} windows", fontsize=5.6); txt(57.5, y + 7.2, f"x {1 - MIX_SINGLE:g}", fontsize=5.8, fontweight="bold", color=C_READ)
    txt(4 + seg_w / 2, yb + 0.9, f"{N_WIN_READ} overlapping {win_s:g}-s windows", fontsize=5, ha="center", color=INK2)
    cv.plot([66, 68.2, 68.2, 66], [y + 1.3, y + 1.3, y + 7.2, y + 7.2], color=INK2, lw=0.6); arrow(68.2, y + 4.25, 73, y + 4.25)
    box(73.6, y + 1.5, 23, 5.5, tint(C_READ, 0.16), ec=C_READ, lw=0.7); txt(85.1, y + 4.25, "probability used", ha="center", fontsize=5.6)
    txt(98, y + 4.25, "no retraining", fontsize=5.2, color=INK2)
    # head: two shifted views of the segment, the difference of their embeddings, and the penalty
    y = rows[2] + 8.6; view_w = seg_w * view_s / seg_s; shift_w = seg_w * shift_s / seg_s
    box(4, y, view_w, 2.6, tint(C_HEAD, 0.5), r=0.3); box(4 + shift_w, y + 3.8, view_w, 2.6, tint(C_HEAD, 0.5), r=0.3); cv.plot([4, 4 + seg_w], [y + 8.2, y + 8.2], color=tint(INK, 0.3), lw=0.5)
    txt(4 + seg_w / 2, y + 9.6, f"two {view_s:g}-s views, {shift_s:g} s apart", fontsize=5, ha="center", color=INK2)
    arrow(35, y + 3.2, 41, y + 3.2); box(41.6, y + 0.4, 24, 5.6, "white", ec=C_HEAD, lw=0.7); txt(53.6, y + 3.2, "embedding difference", ha="center", fontsize=5.4)
    arrow(66, y + 3.2, 72, y + 3.2); box(72.6, y + 0.4, 35, 5.6, tint(C_HEAD, 0.16), ec=C_HEAD, lw=0.7); txt(90.1, y + 3.2, "penalty when the head is refitted", ha="center", fontsize=5.4)

### Panel c: design of the study

A matrix of datasets (rows) and outcomes (columns). A filled circle marks a measure of repeatability (the primary outcome), an open circle a safety check, and a small grey dot an outcome that was not assessed in that dataset.

In [ ]:
def fig1_panel_c(fig):
    panel_letter(fig, 119, 51, "c"); text_mm(fig, 125, 53.2, "This study: outcomes assessed in each dataset", fontsize=6.3, fontweight="bold", va="center", ha="left")
    ax = axes_mm(fig, 143, 84, 32, 27)
    outcomes = ["between adjacent\nsegments", "between\nrecordings", "Discrimination\n(AUROC)", "Response to a\ndocumented change"]
    datasets = ["PTB-XL\n(Germany)", "MIMIC-IV-ECG\n(USA)", "CODE-15%\n(Brazil)", "EchoNext\n(USA)"]
    assessed = np.array([[1, 0, 1, 0], [1, 1, 1, 0], [1, 0, 1, 1], [1, 0, 1, 0]]); SAFETY = INK2
    for r in range(4):
        for c in range(4):
            if assessed[r, c] and c < 2:
                ax.scatter([c], [r], s=46, color=INK, zorder=3)
            elif assessed[r, c]:
                ax.scatter([c], [r], s=40, color="white", edgecolor=SAFETY, linewidth=1.3, zorder=3)
            else:
                ax.scatter([c], [r], s=8, color=GRID, zorder=3)
    ax.set_xlim(-0.5, 3.5); ax.set_ylim(3.5, -0.5); ax.set_xticks(range(4)); ax.xaxis.tick_top(); ax.set_xticklabels(outcomes, rotation=90, ha="center", va="bottom", fontsize=5.4)
    ax.set_yticks(range(4)); ax.set_yticklabels(datasets, fontsize=5.6); ax.tick_params(length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)
    tr = ax.get_xaxis_transform()                                 # headers of the two groups of columns: x in data units, y as a fraction of the height of the axes
    for x0, x1, lab, col in ((-0.4, 1.4, "Repeatability\n(primary)", INK), (1.6, 3.4, "Safety\nchecks", SAFETY)):
        ax.plot([x0, x1], [1.80, 1.80], color=col, lw=0.9, transform=tr, clip_on=False); ax.text((x0 + x1) / 2, 1.83, lab, transform=tr, ha="center", va="bottom", fontsize=5.6, fontweight="bold", color=col, linespacing=0.95)
    text_mm(fig, 121, 116, "Training uses PTB-XL labels and unlabelled MIMIC-IV-ECG recordings\nfrom patients who are in no evaluation set.", fontsize=5.2, ha="left", va="top", color=INK2)

In [ ]:
fig = figure(128)
fig1_panel_a(fig); fig1_panel_b(fig); fig1_panel_c(fig)
save_and_show(fig, "Fig1")

## Fig. 2

**Fig. 2 | Decision changes in three adapted foundation models and their reduction by within-recording remedies.** **a**, Proportion of recordings in which the decision for at least one of the five diagnostic classes differed between the two adjacent 5-s segments, as each remedy is added, in four datasets. **b**, Proportion of pairs of recordings of the same patient, taken 1 min to 24 h apart, in which the decision for at least one diagnostic class differed between the first segments, for pairs with identical machine reports and for all pairs. Points are means of three seeds, and bars are 95% bootstrap confidence intervals. **c**, Relative reduction in decision changes between adjacent segments when each remedy is applied alone to the baseline model and when the three are applied together. Bars are means of the four datasets, and points are the individual datasets.

Data, all from `outcomes_summary.csv`:

- **a** and **b**: `{outcome}_value` with `{outcome}_value_lo` and `{outcome}_value_hi` for the four configurations of `STEPS`; the outcomes are `adjacent_{dataset}` in **a**, and `between_same_report` and `between_all` in **b**.
  The numbers at the two ends of each line are the values at baseline and with the three remedies. The panels of a row share their vertical axis, whose limits are whole numbers at least 1.5 beyond the widest confidence interval.
- **c**: `adjacent_{dataset}_relative_reduction` of the four configurations of `ALONE`. A bar is the mean of the four datasets and the number above it is this mean.
  The reductions of the single remedies do not add up to the reduction of the three together, because their effects overlap.

In [ ]:
def fig2_limits(outcomes, pad=1.5):
    """Whole numbers below the lowest and above the highest confidence limit of the steps, with some room."""
    q = [value_ci(config(m, *key), o) for m in MODELS for _, key in STEPS for o in outcomes]
    return float(np.floor(min(lo for _, lo, _ in q) - pad)), float(np.ceil(max(hi for _, _, hi in q) + pad))


def fig2_panel(ax, outcome, ylim, legend=False, ylabel=None):
    """One outcome as the remedies are added: a line for each foundation model, with the value at baseline (left) and with the three remedies (right) as numbers."""
    x = np.arange(len(STEPS)); first, last = [], []
    for m in MODELS:
        v, lo, hi = (np.array(a) for a in zip(*[value_ci(config(m, *key), outcome) for _, key in STEPS]))
        ax.errorbar(x, v, yerr=[v - lo, hi - v], fmt="none", ecolor=MODEL_COLOR[m], lw=0.6, alpha=0.55, zorder=2)
        ax.plot(x, v, "-", color=MODEL_COLOR[m], lw=1.1, zorder=3, marker=MODEL_MARKER[m], ms=3.1, mfc=MODEL_COLOR[m], mec="white", mew=0.4, label=MODEL_NAMES[m])
        first.append(v[0]); last.append(v[-1])
    gap = 0.052 * (ylim[1] - ylim[0])                             # the numbers of the three models are moved apart where they would overlap
    for m, y0, y1, v0, v1 in zip(MODELS, spread(first, gap), spread(last, gap), first, last):
        ax.text(-0.17, y0, f"{rounded(v0):.1f}", va="center", ha="right", fontsize=5, color=MODEL_COLOR[m]); ax.text(x[-1] + 0.17, y1, f"{rounded(v1):.1f}", va="center", ha="left", fontsize=5, color=MODEL_COLOR[m])
    ax.set_xlim(-0.75, x[-1] + 0.75); ax.set_ylim(*ylim); ax.set_xticks(x); ax.set_xticklabels([lab for lab, _ in STEPS], fontsize=5, rotation=28, ha="right", rotation_mode="anchor")
    ax.set_title(OUTCOME_NAME[outcome], loc="left"); grid(ax, "y")
    if ylabel:
        ax.set_ylabel(ylabel)
    if legend:
        ax.legend(loc="upper right", handlelength=1.6, borderaxespad=0.0)


fig = figure(121)
# a: between adjacent segments, four datasets
panel_letter(fig, 2, 2, "a"); text_mm(fig, 13, 4.2, "Between adjacent segments: first vs second 5-s segment of a recording", fontsize=6.5, fontweight="bold", va="center", ha="left")
_ylim = fig2_limits(OUT_ADJACENT)
for k, o in enumerate(OUT_ADJACENT):
    fig2_panel(axes_mm(fig, 13 + k * 42, 11, 35, 40), o, _ylim, legend=(k == 3), ylabel="Recordings with any decision change (%)" if k == 0 else None)
# b: between recordings, two sets of pairs
panel_letter(fig, 2, 64, "b"); text_mm(fig, 13, 66.2, "Between recordings, 1 min to 24 h apart", fontsize=6.5, fontweight="bold", va="center", ha="left")
_ylim = fig2_limits(OUT_BETWEEN)
for k, o in enumerate(OUT_BETWEEN):
    fig2_panel(axes_mm(fig, 13 + k * 42, 74, 35, 34), o, _ylim, ylabel="Pairs with any decision change (%)" if k == 0 else None)
# c: each remedy alone and the three together, relative reduction between adjacent segments
panel_letter(fig, 99, 64, "c"); text_mm(fig, 104, 66.2, "Each remedy alone and the three together", fontsize=6.5, fontweight="bold", va="center", ha="left")
ax = axes_mm(fig, 112, 74, 63, 34); bar_width = 0.19; _highest = 0.0
for gi, m in enumerate(MODELS):
    for bi, (lab, key, col) in enumerate(ALONE):
        rel = np.array([config(m, *key)[f"{o}_relative_reduction"] for o in OUT_ADJACENT], float); xc = gi + (bi - 1.5) * (bar_width + 0.02)
        ax.bar(xc, rel.mean(), width=bar_width, color=col, edgecolor="white", linewidth=0.5, zorder=2, label=lab if gi == 0 else None)
        ax.scatter(np.full(len(rel), xc) + np.linspace(-0.05, 0.05, len(rel)), rel, s=5, color="white", edgecolor=INK2, linewidth=0.45, zorder=3)
        ax.text(xc, max(rel.max(), rel.mean()) + 1.6, f"{rounded(rel.mean(), 0):.0f}", ha="center", va="bottom", fontsize=5, color=INK)
        _highest = max(_highest, rel.max())
ax.set_xticks(range(len(MODELS))); ax.set_xticklabels([MODEL_NAMES[m] for m in MODELS]); ax.set_xlim(-0.55, len(MODELS) - 0.45)
ax.set_ylim(0, float(np.ceil((_highest + 20) / 10) * 10))        # room for the legend above the highest point
ax.set_ylabel("Relative reduction in decision changes\nbetween adjacent segments (%)"); grid(ax, "y")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.02), ncol=2, handlelength=0.9, columnspacing=1.2, borderaxespad=0.0)
save_and_show(fig, "Fig2")

## Fig. 3

**Fig. 3 | Comparison of the three remedies with an ensemble and with Monte Carlo dropout in ECGFounder.** Paired differences from the baseline in PTB-XL with an ensemble of ten baseline models, Monte Carlo dropout (baseline trained with dropout at a rate of 0.3; mean of 25 passes) and the three remedies: decision change between adjacent segments, in percentage points (left), and five-diagnosis AUROC (right). Points are means of three seeds, and bars are 95% bootstrap confidence intervals. Filled symbols mark intervals that exclude zero. All layers were fine-tuned. AUROC, area under the receiver operating characteristic curve.

Data: three rows of `outcomes_summary.csv` for ECGFounder, with the columns `adjacent_ptbxl_diff` and `auroc_ptbxl_diff` and their confidence limits:

- the ensemble: training `ensemble10` (one predictor, compared with the baseline model of each of the three seeds);
- Monte Carlo dropout: the row with the reading `mcdropout` (the dropout rate with the highest AUROC in the validation fold, notebook 05);
- the three remedies.

In [ ]:
_ensemble = config("ecgfounder", f"ensemble{len(ENSEMBLE_SEEDS)}", "model", "single")
_mc = SUM[(SUM["model"] == "ecgfounder") & (SUM["reading"] == "mcdropout")]
assert len(_mc) == 1, "one row with Monte Carlo dropout is expected"
print("Monte Carlo dropout: dropout rate", float(_mc["dropout_rate"].iloc[0]))
FIG3_ROWS = [("Ensemble of ten baseline models", _ensemble, C_BASE), ("Monte Carlo dropout", _mc.iloc[0], C_BASE), ("Three remedies", config("ecgfounder", *REMEDIES), INK)]

fig = figure(39)
_height = 6.5 * len(FIG3_ROWS)                                    # height of the plotting area: 6.5 mm for each row
for k, (o, x0, w, xlabel) in enumerate((("adjacent_ptbxl", 37, 44, "Difference in decision change between\nadjacent segments (percentage points)"), ("auroc_ptbxl", 87, 32, "Difference in AUROC"))):
    ax = axes_mm(fig, x0, 3.0, w, _height); ends = [0.0]
    for j, (lab, r, col) in enumerate(FIG3_ROWS):
        d, lo, hi, sig = diff_ci(r, o); ends += [lo, hi]
        ax.errorbar([d], [j], xerr=[[d - lo], [hi - d]], fmt="o", ms=3.6, color=col, mfc=col if sig else "white", mec=col, mew=0.7, ecolor=col, lw=0.8, zorder=3)
    ax.axvline(0, color=AXIS, lw=0.5, zorder=1); ax.set_xlim(*axis_limits(("Fig3", o), ends)); ax.set_ylim(len(FIG3_ROWS) - 0.5, -0.5)
    ax.set_yticks(range(len(FIG3_ROWS))); ax.set_yticklabels([lab for lab, *_ in FIG3_ROWS] if k == 0 else []); ax.tick_params(axis="y", length=0); ax.set_xlabel(xlabel); grid(ax, "x")
    for j in range(1, len(FIG3_ROWS)):
        ax.axhline(j - 0.5, color=GRID, lw=0.4, zorder=0)
    if k == 1:
        ax.xaxis.set_major_locator(MultipleLocator(0.005))        # a tick every 0.005: the labels of a finer grid would touch each other
    tr = ax.get_xaxis_transform(); y_text = -(11.6 if k == 0 else 9.0) / _height       # direction of the difference, below the label of the axis
    if k == 0:
        ax.text(0, y_text, "← Fewer decision changes than baseline  ", transform=tr, ha="right", va="top", fontsize=5.5, color=INK2, clip_on=False)
    else:
        ax.text(0, y_text, "← Lower than  \nbaseline  ", transform=tr, ha="right", va="top", fontsize=5.5, color=INK2, clip_on=False, linespacing=1.0)
        ax.text(0, y_text, "  Higher than →\n  baseline", transform=tr, ha="left", va="top", fontsize=5.5, color=INK2, clip_on=False, linespacing=1.0)
save_and_show(fig, "Fig3")

## Fig. 4

**Fig. 4 | Discrimination and response to documented change with the three remedies in three foundation models.** Paired differences, calculated as the three remedies minus the baseline of the same model, so that values to the right of zero favour the three remedies and values to the left favour the baseline. **a**, Five measures of AUROC. **b**, Direction agreement in pairs with a documented change in CODE-15%. Bars are 95% bootstrap confidence intervals, and filled symbols mark intervals that exclude zero. AUROC, area under the receiver operating characteristic curve.

Data: the row of the three remedies of each foundation model in `outcomes_summary.csv`, with `{outcome}_diff`, its confidence limits and `{outcome}_sig` for the five AUROCs (**a**) and the two direction agreements (**b**).
Within each measure the three models are drawn one below the other.

In [ ]:
fig = figure(67)
_offset = dict(zip(MODELS, (-0.25, 0.0, 0.25)))                   # vertical position of each model within a row
for k, (outcomes, key, letter_x, x0, w, h, xlabel, title, y_text) in enumerate((
        (OUT_AUROC, "auroc", 2.0, 40.5, 40, 44, "Difference in AUROC", "Discrimination", 9.2),
        (OUT_DIRECTION, "direction", 87.0, 123, 40, 18, "Difference in direction agreement\n(percentage points)", "Response to documented change", 11.8))):
    panel_letter(fig, letter_x, 2, "ab"[k])
    ax = axes_mm(fig, x0, 8, w, h); ends = []
    for j, o in enumerate(outcomes):
        for m in MODELS:
            d, lo, hi, sig = diff_ci(config(m, *REMEDIES), o); ends += [lo, hi]
            ax.errorbar([d], [j + _offset[m]], xerr=[[d - lo], [hi - d]], fmt=MODEL_MARKER[m], ms=3.4, color=MODEL_COLOR[m], mfc=MODEL_COLOR[m] if sig else "white", mec=MODEL_COLOR[m], mew=0.7, ecolor=MODEL_COLOR[m], lw=0.7, zorder=3)
    ax.axvline(0, color=AXIS, lw=0.5, zorder=1); ax.set_yticks(range(len(outcomes))); ax.set_yticklabels([OUTCOME_NAME[o] for o in outcomes]); ax.set_ylim(len(outcomes) - 0.45, -0.55)
    ax.set_xlim(*axis_limits(("Fig4", key), ends)); ax.set_xlabel(xlabel); ax.set_title(title, loc="left"); grid(ax, "x")
    tr = ax.get_xaxis_transform()                                 # direction of the difference, below the label of the axis (y_text mm below the axis)
    ax.text(0, -y_text / h, "← Favours baseline  ", transform=tr, ha="right", va="top", fontsize=5.5, color=INK2, clip_on=False)
    ax.text(0, -y_text / h, "  Favours three remedies →", transform=tr, ha="left", va="top", fontsize=5.5, color=INK2, clip_on=False)
    for j in range(1, len(outcomes)):
        ax.axhline(j - 0.5, color=GRID, lw=0.4, zorder=0)
    if k == 1:
        handles = [Line2D([], [], ls="none", marker=MODEL_MARKER[m], ms=3.4, color=MODEL_COLOR[m], mec=MODEL_COLOR[m], mew=0.7, label=MODEL_NAMES[m]) for m in MODELS]
        ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.0, -1.05), handletextpad=0.3, ncol=1)
save_and_show(fig, "Fig4")

## Fig. 5

**Fig. 5 | Contribution and cost of the penalized head in three foundation models.** **a**, Relative reduction in decision changes from baseline with distillation and mixed reading (open symbols) and with the penalized head added (filled symbols), between adjacent segments (left; mean of the four datasets) and between recordings with identical machine reports (right). The length of each line is the part added by the head. **b**, Relative reduction in decision changes between adjacent segments (mean of the four datasets) against the difference from baseline in the five-diagnosis AUROC in PTB-XL, for the distilled model with mixed reading as the penalty weight increases from 30 to 1,000 (numbers). **c**, As **b**, with the difference from baseline in direction agreement for conduction disturbance in CODE-15%, in percentage points, on the vertical axis. The lines run from a weight of 30 to 1,000 as in **b**. In **b** and **c**, large symbols mark the weight used in this study (300), and open symbols the same model without the penalized head. Points are means of three seeds, and bars are 95% bootstrap confidence intervals of the paired difference from the baseline. AUROC, area under the receiver operating characteristic curve.

Data: the rows of the distilled model with mixed reading in `outcomes_summary.csv`, with the model head (`model`, without the penalized head) and with the penalized head at each weight (`pen30`, `pen100`, `pen300`, `pen1000`):

- **a**: `adjacent_mean4_relative_reduction` (left) and `between_same_report_relative_reduction` (right), with the model head and with the head of the final setting;
- **b** and **c**: `adjacent_mean4_relative_reduction` on the horizontal axis; `auroc_ptbxl_diff` (**b**) and `direction_cd_diff` (**c**) with their confidence limits on the vertical axis.

All reductions in this figure are relative (% of the value at baseline), whereas the differences on the vertical axes of **b** and **c** are absolute.
The axes of **b** and **c** have fixed limits, and the place of each number in **b** is set by hand for these data (`FIG5_NUMBER`, a shift from the point), so that no number covers a line or a bar.

In [ ]:
FIG5_WEIGHTS = [g for g in PENALTY_WEIGHTS if g > 0]              # 30, 100, 300, 1000; weight 0 (refitted without a penalty) is not part of this figure
ADJ, REC = "adjacent_mean4_relative_reduction", "between_same_report_relative_reduction"
with_head = lambda m, g: config(m, "student", f"pen{g}", MIXED)
without_head = lambda m: config(m, "student", "model", MIXED)

fig = figure(118)
# a: what the head adds to distillation and mixed reading
panel_letter(fig, 2, 2, "a"); text_mm(fig, 8, 4.2, "Reduction added by the penalized head", fontsize=6.5, fontweight="bold", va="center", ha="left")
text_mm(fig, 8, 8.6, "relative reduction from baseline", fontsize=5.5, color=INK2, va="center", ha="left")
_handles = [Line2D([], [], ls="none", marker="o", ms=4.2, mfc="white", mec=C_BASE, mew=0.9, label="Distillation and mixed reading"),
            Line2D([], [], ls="none", marker="o", ms=4.6, mfc=C_HEAD, mec="white", mew=0.5, label="With the penalized head added")]
for j, (x0, col, sub) in enumerate(((26, ADJ, "Between adjacent segments"), (128, REC, "Between recordings (identical machine reports)"))):
    ax = axes_mm(fig, x0, 19, 42, 15)
    for i, m in enumerate(MODELS):
        a0, a1 = float(without_head(m)[col]), float(with_head(m, PENALTY_WEIGHT)[col])
        ax.plot([a0, a1], [i, i], color=C_HEAD, lw=2.2, solid_capstyle="butt", zorder=2)
        ax.plot([a0], [i], "o", ms=4.2, mfc="white", mec=C_BASE, mew=0.9, zorder=3); ax.plot([a1], [i], "o", ms=4.6, mfc=C_HEAD, mec="white", mew=0.5, zorder=4)
        ax.text(a0 - 2.4, i, f"{a0:.0f}", ha="right", va="center", fontsize=5.5, color=INK2); ax.text(a1 + 2.4, i, f"{a1:.0f}", ha="left", va="center", fontsize=5.5, color=INK)
    ax.set_title(sub, loc="left", fontsize=5.5, color=INK2, fontweight="bold", pad=2)
    ax.set_yticks(range(len(MODELS))); ax.set_yticklabels([MODEL_NAMES[m] for m in MODELS]); ax.tick_params(axis="y", length=0); ax.set_ylim(len(MODELS) - 0.4, -0.6); ax.set_xlim(-8, 66); ax.set_xticks([0, 20, 40, 60])
    ax.set_xlabel("Relative reduction in decision changes (%)"); grid(ax, "x"); ax.spines["left"].set_visible(False)
    if j == 0:
        ax.legend(handles=_handles, loc="center left", bbox_to_anchor=(1.06, 0.5), ncol=1, handletextpad=0.3)

# b and c: the two costs as the penalty weight grows
# where each weight is written in b: (model, weight) -> (shift to the right and upwards from the point, in points; horizontal and vertical alignment)
FIG5_NUMBER = {("ecgfounder", 30): (0, 7.62, "center", "bottom"), ("ecgfounder", 100): (0, 7.49, "center", "bottom"), ("ecgfounder", 300): (4.5, 3.0, "left", "bottom"), ("ecgfounder", 1000): (4.0, 0, "left", "center"),
               ("hubert", 30): (0, 7.17, "center", "bottom"), ("hubert", 100): (0, 7.06, "center", "bottom"), ("hubert", 300): (5.0, 2.0, "left", "bottom"), ("hubert", 1000): (0, -10.97, "center", "top"),
               ("ecgfm", 30): (0, -6.04, "center", "top"), ("ecgfm", 100): (-4.0, -1.0, "right", "top"), ("ecgfm", 300): (-5.0, 0, "right", "center"), ("ecgfm", 1000): (-4.0, 0, "right", "center")}
_panels = [("b", 2, 24, "auroc_ptbxl", "Cost in discrimination", "five-diagnosis AUROC, PTB-XL; numbers, penalty weight", "Difference in AUROC", (-0.035, 0.013), True),
           ("c", 94, 112, "direction_cd", "Cost in the response to change", f"conduction disturbance, CODE-15%; lines from a weight of {FIG5_WEIGHTS[0]:,} to {FIG5_WEIGHTS[-1]:,}, as in b", "Difference in direction agreement\n(percentage points)", (-5.6, 3.2), False)]
for letter, letter_x, x0, o, title, sub, ylabel, ylim, numbers in _panels:
    panel_letter(fig, letter_x, 50, letter); text_mm(fig, letter_x + 6, 52.2, title, fontsize=6.5, fontweight="bold", va="center", ha="left"); text_mm(fig, letter_x + 6, 56.6, sub, fontsize=5.5, color=INK2, va="center", ha="left")
    ax = axes_mm(fig, x0, 62, 64, 40)
    for m in MODELS:
        col, marker = MODEL_COLOR[m], MODEL_MARKER[m]; q = [with_head(m, g) for g in FIG5_WEIGHTS]
        x = np.array([float(r[ADJ]) for r in q]); v, lo, hi = (np.array(a) for a in zip(*[diff_ci(r, o)[:3] for r in q]))
        ax.errorbar(x, v, yerr=[v - lo, hi - v], ls="-", color=col, lw=1.0, marker=marker, ms=3.2, mec="white", mew=0.4, ecolor=col, elinewidth=0.6, zorder=3)
        k = FIG5_WEIGHTS.index(PENALTY_WEIGHT); ax.plot([x[k]], [v[k]], ls="none", marker=marker, ms=6.2, color=col, mec="white", mew=0.6, zorder=5)       # the weight of the final setting
        r0 = without_head(m); d0, lo0, hi0, _ = diff_ci(r0, o)
        ax.errorbar([float(r0[ADJ])], [d0], yerr=[[d0 - lo0], [hi0 - d0]], ls="none", marker=marker, ms=3.6, mfc="white", mec=col, mew=0.8, ecolor=col, elinewidth=0.6, zorder=4)
        if numbers:
            for g, xx, vv in zip(FIG5_WEIGHTS, x, v):
                dx, dy, ha, va = FIG5_NUMBER.get((m, g), (0, 8.0, "center", "bottom"))
                ax.annotate(f"{g:,}", xy=(xx, vv), xytext=(dx, dy), textcoords="offset points", ha=ha, va=va, fontsize=5, color=INK2)
    ax.axhline(0, color=AXIS, lw=0.5, zorder=1); ax.set_xlim(8, 64); ax.set_ylim(*ylim); grid(ax, "y")
    ax.set_xlabel("Relative reduction in decision changes between adjacent segments (%)"); ax.set_ylabel(ylabel)
    ax.text(0.02, 0.975, "↑ Higher than baseline", transform=ax.transAxes, ha="left", va="top", fontsize=5, color=INK2); ax.text(0.02, 0.025, "↓ Lower than baseline", transform=ax.transAxes, ha="left", va="bottom", fontsize=5, color=INK2)
    if letter == "b":
        handles = [Line2D([], [], ls="-", lw=1.0, marker=MODEL_MARKER[m], ms=3.2, color=MODEL_COLOR[m], mec="white", mew=0.4, label=MODEL_NAMES[m]) for m in MODELS]
        handles += [Line2D([], [], ls="none", marker="o", ms=3.6, mfc="white", mec=INK2, mew=0.8, label="Without the penalized head"),
                    Line2D([], [], ls="none", marker="o", ms=6.2, color=INK2, mec="white", mew=0.6, label=f"Penalty weight used ({PENALTY_WEIGHT:,})")]
        ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(-0.02, -0.2), ncol=5, handletextpad=0.4, columnspacing=1.2)
save_and_show(fig, "Fig5")

## Supplementary Fig. 1

**Supplementary Fig. 1 | Flow of recordings into the training and evaluation sets.** Exclusions based on statements and metadata were applied before sampling, and the waveform check was applied to each candidate as it was examined. Stacked boxes without arrows are sets drawn from the same parent box. Sets used in training are shown in blue and evaluation sets in green.

Data: `SET_COUNTS`, the file in which notebook 02 collected the numbers while it built the sets (its layout is documented in the last section of that notebook).
It has one entry for each dataset (`ptbxl`, `mimic`, `code15`, `echonext`) with the recordings and patients of the dataset and the exclusions before sampling, and under `sets` one entry for each sampled set with

- `candidates`: patients from whom the set could be drawn;
- `examined`: patients examined, in random order, until the set was complete;
- `selected`: patients in the set;
- `rejected`: patients who were examined and not selected, by reason: `waveform_check` (no recording, or no pair, was left after the waveform check) and `identical_file` (no pair was left after a recording with an identical waveform file was dropped).

Every number of the diagram is one of these entries.

In [ ]:
if not SET_COUNTS.exists():
    raise FileNotFoundError("set_counts.json (SET_COUNTS) is missing: run notebook 02 first")
COUNTS = json.loads(SET_COUNTS.read_text(encoding="utf-8"))
print({d: list(COUNTS[d].get("sets", {})) for d in COUNTS})

### Drawing

One column for each dataset. A box is placed by its distance from the top and its height in millimetres. An arrow leads from each box to the next one, except to a box marked with `*`,
which is drawn from the same parent as the box above it (stacked boxes without an arrow).

In [ ]:
def draw_supp_fig1(counts):
    """Flow diagram from the numbers of SET_COUNTS."""
    ptb, mimic, code, echo = counts["ptbxl"], counts["mimic"], counts["code15"], counts["echonext"]
    pool, mimic_sets, code_sets, echo_sets = mimic["pool"], mimic["sets"], code["sets"], echo["sets"]
    num = lambda v: f"{int(v):,}"
    r_waves = {3: "three"}.get(MIN_R_WAVES, MIN_R_WAVES)          # criterion of the waveform check (notebook 00), written as a word
    failed = lambda s: num(s["rejected"].get("waveform_check", 0))                     # patients rejected by the waveform check
    examined = lambda s: f"({num(s['examined'])} examined; {failed(s)} failed the waveform check)"

    def pairs(title, s):
        identical = s["rejected"].get("identical_file", 0)                             # patients rejected because of an identical waveform file
        tail = f"{failed(s)} failed the waveform\ncheck, {num(identical)} had identical files)" if identical else f"{failed(s)} failed the waveform check)"
        return f"{title}\n{num(s['selected'])} of {num(s['candidates'])} candidate patients\n({num(s['examined'])} examined; {tail}"

    height = 174; fig = figure(height); ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, WIDTH_MM); ax.set_ylim(height, 0); ax.axis("off")      # one canvas in millimetres
    face = {"src": "#f1f0ec", "excl": "white", "mid": "white", "train": tint(C_TRAIN, 0.16), "eval": tint(C_READ, 0.16)}

    def box(x, y, w, h, text, kind):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0,rounding_size=1.0", fc=face[kind], ec=AXIS if kind != "excl" else MUTED, lw=0.6, ls=(0, (2.5, 1.8)) if kind == "excl" else "-"))
        lines = text.split("\n"); line_height = 2.7; y0 = y + h / 2 - (len(lines) - 1) * line_height / 2
        for i, t in enumerate(lines):                             # the first line of a box is its title
            ax.text(x + w / 2, y0 + i * line_height, t, ha="center", va="center", fontsize=5.3, fontweight="bold" if i == 0 else "normal", color=INK if kind != "excl" else INK2)

    def arrow(x, y0, y1):
        ax.annotate("", xy=(x, y1), xytext=(x, y0), arrowprops=dict(arrowstyle="-|>", lw=0.6, color=AXIS, shrinkA=0, shrinkB=0, mutation_scale=5))

    # each box: (kind, distance from the top, height, text); a kind ending with * is stacked below the box above it, without an arrow
    columns = [
        [("src", 6, 12, f"PTB-XL (Germany)\n{num(ptb['recordings'])} recordings\n{num(ptb['patients'])} patients"),
         ("excl", 22, 10, f"Excluded before sampling\n{num(ptb['excluded_no_diagnostic_statement'])} without a diagnostic statement"), ("mid", 36, 7, f"{num(ptb['recordings_with_diagnostic_statement'])} recordings"),
         ("train", 48, 10, f"Training (folds 1-8)\n{num(ptb['train'])} recordings"), ("train*", 59.2, 12.5, f"Validation (fold 9)\n{num(ptb['val'])} recordings\ncheckpoints, thresholds"),
         ("eval*", 72.9, 10, f"Test (fold 10)\n{num(ptb['test'])} recordings"),
         ("excl", 87.5, 12.5, f"Excluded\n{num(ptb['test_excluded_by_statement'])} with premature beats, pacing,\nbigeminy or trigeminy"), ("eval", 104.5, 10, f"Subset for adjacent segments\n{num(ptb['test_adjacent'])} recordings")],
        [("src", 6, 12, f"MIMIC-IV-ECG (USA)\n{num(mimic['recordings'])} recordings\n{num(mimic['patients'])} patients"),
         ("excl", 22, 10, f"Excluded before sampling\n{num(mimic['excluded_by_statement'])} with statements of poor quality,\npacing or premature beats"),
         ("mid", 36, 7, f"{num(mimic['recordings_after_statement_exclusion'])} recordings, {num(mimic['patients_after_statement_exclusion'])} patients"),
         ("train", 48, 15, f"Unlabelled training pool\n{num(pool['patients_drawn'])} random patients\n{num(pool['recordings'])} recordings of {num(pool['patients'])} patients\n"
                           f"({num(pool['recordings_examined'])} examined; {num(pool['recordings_failed_waveform_check'])} failed the waveform check)"),
         ("mid*", 64.2, 9, f"Candidates for evaluation\n{num(mimic['evaluation_candidate_patients'])} patients"),
         ("eval", 78, 12.5, f"Test\n{num(mimic_sets['mimic_test']['selected'])} patients, one recording each\n{examined(mimic_sets['mimic_test'])}"),
         ("mid*", 91.7, 15, f"Candidates for pairs\n{num(mimic['pair_candidate_patients_before_time_exclusion'])} patients with two successive\nrecordings 1 min to 24 h apart (for each\npatient, the pair with the shortest interval)"),
         ("excl", 111.5, 12.5, f"Excluded from pairs\n{num(mimic['recordings_excluded_from_pairs_time_mismatch'])} recordings with different times\nin the two metadata files"),
         ("eval", 128.8, 15, pairs("Pairs, all", mimic_sets["mimic_pairs_all"])), ("eval*", 145, 15, pairs("Pairs, identical machine report", mimic_sets["mimic_pairs_same_report"]))],
        [("src", 6, 12, f"CODE-15% (Brazil)\n{num(code['exams'])} exams\n{num(code['patients'])} patients"),
         ("excl", 22, 10, f"Excluded before sampling\n{num(code['excluded_shorter_than_10_s'])} exams without a\ncomplete 10-s tracing"),
         ("mid", 36, 7, f"{num(code['exams_after_exclusion'])} exams, {num(code['patients_after_exclusion'])} patients"),
         ("eval", 48, 12.5, f"Test\n{num(code_sets['code15_test']['selected'])} patients, one exam each\n{examined(code_sets['code15_test'])}"),
         ("eval*", 61.7, 12.5, f"Conduction-change pairs\n{num(code_sets['code15_cd_change']['selected'])} of {num(code_sets['code15_cd_change']['candidates'])} candidate patients\n{examined(code_sets['code15_cd_change'])}"),
         ("eval*", 75.4, 12.5, f"Normal-change pairs\n{num(code_sets['code15_normal_change']['selected'])} of {num(code_sets['code15_normal_change']['candidates'])} candidate patients\n{examined(code_sets['code15_normal_change'])}")],
        [("src", 6, 12, f"EchoNext (USA)\n{num(echo['recordings'])} ECGs\n{num(echo['patients'])} patients"),
         ("mid", 36, 7, f"{num(echo['recordings'])} ECGs, {num(echo['patients'])} patients"),
         ("eval", 48, 12.5, f"Test\n{num(echo_sets['echonext_test']['selected'])} patients, one ECG each\n{examined(echo_sets['echonext_test'])}")]]
    box_width = 42.5
    for x, column in zip([2, 46.5, 91, 135.5], columns):
        previous = None
        for kind, y, h, text in column:
            stacked = kind.endswith("*"); kind = kind.rstrip("*")
            box(x, y, box_width, h, text, kind)
            if previous is not None and not stacked:
                arrow(x + box_width / 2, previous, y)
            previous = y + h
    for i, (kind, lab) in enumerate((("train", "Used in training"), ("eval", "Evaluation set"), ("excl", "Exclusion"))):                         # key to the kinds of box
        ax.add_patch(FancyBboxPatch((93 + i * 30, 96), 5, 3.2, boxstyle="round,pad=0,rounding_size=0.6", fc=face[kind], ec=AXIS if kind != "excl" else MUTED, lw=0.6, ls=(0, (2.5, 1.8)) if kind == "excl" else "-"))
        ax.text(100 + i * 30, 97.6, lab, fontsize=5.6, va="center", ha="left")
    notes = (f"Waveform check: at least {r_waves} R waves in each 5-s half of the filtered lead II, applied to each candidate as it was examined in random order, until a set was complete.\n"
             "Patients in the training pool are in no evaluation set. Evaluation sets of one dataset were sampled independently and may share patients.")
    ax.text(2, 164, notes, fontsize=5.2, va="top", ha="left", color=INK2, linespacing=1.45)
    return fig


save_and_show(draw_supp_fig1(COUNTS), "SuppFig1")

## Supplementary Fig. 2

**Supplementary Fig. 2 | Alternatives for each of the three remedies in ECGFounder.** Decision change between adjacent segments in PTB-XL (top) and difference from baseline in the five-diagnosis AUROC in PTB-XL (bottom). **a**, Training methods, read once with the model head: labels only (baseline, open symbol), training on 2.5-s windows, a direct agreement loss between shifted views of the segment, distillation of the window average of the baseline model itself (self-distillation), and distillation from a teacher trained on 2.5-s windows. **b**, Weight of the window average in mixed reading, for the distilled model with the penalized head. A weight of 0 is a single read of the segment. **c**, Head of the distilled model with mixed reading: the model head, and heads refitted with the penalty weight γ (0, refitted without a penalty). Grey bands mark the options used in this study. Points are means of three seeds, and bars are 95% bootstrap confidence intervals of the paired difference from the baseline. In the bottom row, values above zero indicate a higher AUROC than at baseline. All layers were fine-tuned. AUROC, area under the receiver operating characteristic curve.

Data: rows of ECGFounder in `outcomes_summary.csv`. Each panel varies one remedy:

| Panel | Varied | Rows |
|---|---|---|
| **a** | training | `baseline`, `windows`, `agreement`, `selfdistill`, `student`; head `model`, reading `single` |
| **b** | weight of the window average | `student`, head of the final setting; readings `single` (0), `mixed0.25`, `mixed0.5`, `mixed0.75`, `mixed0.9`, `windows` (1) |
| **c** | head | `student`, mixed reading; heads `model`, `pen0`, `pen30`, `pen100`, `pen300`, `pen1000` |

Top row: `adjacent_ptbxl_value`. The bar around each value is the confidence interval of its **paired difference from the baseline** (`adjacent_ptbxl_diff_lo`, `_diff_hi`), drawn around the value;
it shows how sure the difference from the baseline is, and the baseline itself has no bar. Bottom row: `auroc_ptbxl_diff` with its confidence limits; the baseline is the zero line.
In **c** the model head is not joined to the refitted heads by a line, because it is not a point on the scale of the penalty weight.

In [ ]:
S2_MODEL = "ecgfounder"
S2_TRAINING = [("baseline", "Labels\nonly"), ("windows", "2.5-s\nwindows"), ("agreement", "Agreement\nloss"), ("selfdistill", "Self-\ndistillation"), ("student", "Distillation")]
S2_READING = [(0.0, "single", "0"), (0.25, "mixed0.25", "0.25"), (0.5, "mixed0.5", "0.5"), (0.75, "mixed0.75", "0.75"), (0.9, "mixed0.9", "0.9"), (1.0, "windows", "1")]
S2_HEADS = [(0, "model", "Model\nhead")] + [(i + 1, f"pen{g}", f"{g:,}") for i, g in enumerate(PENALTY_WEIGHTS)]
# the points of each panel: (position on the horizontal axis, row of the summary table)
S2_POINTS = {"a": [(i, config(S2_MODEL, k, "model", "single")) for i, (k, _) in enumerate(S2_TRAINING)],
             "b": [(w, config(S2_MODEL, "student", HEAD, r)) for w, r, _ in S2_READING],
             "c": [(x, config(S2_MODEL, "student", h, MIXED)) for x, h, _ in S2_HEADS]}
S2_TICKS = {"a": (list(range(len(S2_TRAINING))), [t for _, t in S2_TRAINING]), "b": ([w for w, _, _ in S2_READING], [t for _, _, t in S2_READING]), "c": ([x for x, _, _ in S2_HEADS], [t for _, _, t in S2_HEADS])}
S2_ROWS = [("adjacent_ptbxl", "Decision change,\nadjacent segments,\nPTB-XL (%)"), ("auroc_ptbxl", "Difference in AUROC,\nPTB-XL")]
_final = {"a": len(S2_TRAINING) - 1, "b": 1 - MIX_SINGLE, "c": 1 + PENALTY_WEIGHTS.index(PENALTY_WEIGHT)}       # position of the option of the final setting in each panel
S2_BAND = {"a": (_final["a"] - 0.5, _final["a"] + 0.5), "b": (_final["b"] - 0.05, _final["b"] + 0.05), "c": (_final["c"] - 0.4, _final["c"] + 0.4)}


def s2_bar(r, outcome):
    """Half-widths (down, up) of the 95% CI of the paired difference from the baseline; zero for the baseline itself, which has no difference."""
    d, lo, hi, _ = diff_ci(r, outcome)
    return (d - lo, hi - d) if np.isfinite([d, lo, hi]).all() else (0.0, 0.0)


def s2_limits(outcome):
    """The same vertical axis for the three panels of a row."""
    rows = [r for pts in S2_POINTS.values() for _, r in pts]
    if outcome.startswith("auroc"):
        ends = [float(r[f"{outcome}_diff{s}"]) for r in rows for s in ("_lo", "_hi") if np.isfinite(r[f"{outcome}_diff{s}"])] + [0.0]
    else:
        ends = [float(r[f"{outcome}_value"]) + sign * e for r in rows for sign, e in zip((-1, 1), s2_bar(r, outcome))]
    return axis_limits(("SuppFig2", outcome), ends)


fig = figure(16 + 29 * len(S2_ROWS) + 11)
_columns = {"a": (25, 50), "b": (84, 40), "c": (133, 44)}        # left edge and width of the plotting areas (mm)
_title = {"a": ("Training", "model head, single read"), "b": ("Reading", "distilled model, penalized head"), "c": ("Head", "distilled model, mixed reading")}
_color = {"a": C_TRAIN, "b": C_READ, "c": C_HEAD}                # colour of the remedy that is varied
_xlim = {"a": (-0.6, len(S2_TRAINING) - 0.4), "b": (-0.08, 1.08), "c": (-0.6, len(S2_HEADS) - 0.5)}
for p, (x0, w) in _columns.items():
    panel_letter(fig, x0 - (23 if p == "a" else 5), 2, p); text_mm(fig, x0 + (-17 if p == "a" else 1), 4.2, _title[p][0], fontsize=6.5, fontweight="bold", va="center", ha="left")
    text_mm(fig, x0 + (-17 if p == "a" else 1), 8.6, _title[p][1], fontsize=5.5, color=INK2, va="center", ha="left")
    for i, (o, ylabel) in enumerate(S2_ROWS):
        ax = axes_mm(fig, x0, 16 + 29 * i, w, 24); col = _color[p]; pts = S2_POINTS[p]
        ax.axvspan(*S2_BAND[p], color=GRID, alpha=0.6, lw=0, zorder=0)
        if o.startswith("auroc"):                                 # differences from the baseline; the baseline itself is the zero line
            q = [(x, r) for x, r in pts if np.isfinite(r[f"{o}_diff"])]
            if p == "c":                                          # the model head apart from the line of the refitted heads
                for x, r in [(x, r) for x, r in q if x == 0]:
                    d, lo, hi, _ = diff_ci(r, o)
                    ax.errorbar([x], [d], yerr=[[d - lo], [hi - d]], fmt="o", ms=3.0, color=col, mec="white", mew=0.4, ecolor=col, lw=0.6, zorder=3)
                q = [(x, r) for x, r in q if x > 0]
            xs = np.array([x for x, _ in q], float); v, lo, hi = (np.array(a) for a in zip(*[diff_ci(r, o)[:3] for _, r in q]))
            ax.errorbar(xs, v, yerr=[v - lo, hi - v], ls="-" if p != "a" else "none", color=col, lw=1.0, marker="o", ms=3.0, mec="white", mew=0.4, ecolor=col, elinewidth=0.6, zorder=3)
        elif p == "a":                                            # values; the baseline as an open symbol in the colour of the baseline, without a bar
            for x, r in pts:
                c = C_BASE if x == 0 else col; e = s2_bar(r, o) if x > 0 else (0.0, 0.0)
                ax.errorbar([x], [r[f"{o}_value"]], yerr=[[e[0]], [e[1]]], ls="none", marker="o", ms=3.2, color=c, mfc="white" if x == 0 else c, mec=c if x == 0 else "white", mew=0.7 if x == 0 else 0.4, ecolor=c, elinewidth=0.6, zorder=3)
        else:
            for x, r in [(x, r) for x, r in pts if p == "c" and x == 0]:
                ax.errorbar([x], [r[f"{o}_value"]], yerr=[[s2_bar(r, o)[0]], [s2_bar(r, o)[1]]], ls="none", marker="o", ms=3.0, color=col, mec="white", mew=0.4, ecolor=col, elinewidth=0.6, zorder=3)
            q = [(x, r) for x, r in pts if not (p == "c" and x == 0)]
            ax.errorbar(np.array([x for x, _ in q], float), [r[f"{o}_value"] for _, r in q], yerr=[[s2_bar(r, o)[0] for _, r in q], [s2_bar(r, o)[1] for _, r in q]], ls="-", color=col, lw=1.0, marker="o", ms=3.0,
                        mec="white", mew=0.4, ecolor=col, elinewidth=0.6, zorder=3)
        if o.startswith("auroc"):
            ax.axhline(0, color=AXIS, lw=0.5, zorder=1)
            if p == "a":                                          # direction of the difference, written in the empty column of the baseline
                ax.annotate("Higher than\nbaseline ↑", xy=(0, 0), xytext=(0, 2.5), textcoords="offset points", ha="center", va="bottom", fontsize=5, color=INK2, linespacing=1.0)
                ax.annotate("Lower than\nbaseline ↓", xy=(0, 0), xytext=(0, -2.5), textcoords="offset points", ha="center", va="top", fontsize=5, color=INK2, linespacing=1.0)
        ax.set_ylim(*s2_limits(o)); ax.set_xlim(*_xlim[p]); grid(ax, "y")
        ax.set_xticks(S2_TICKS[p][0]); ax.set_xticklabels(S2_TICKS[p][1] if i == len(S2_ROWS) - 1 else [])
        if p == "a":
            ax.set_ylabel(ylabel)
        else:
            ax.set_yticklabels([])
        if i == 0:
            ax.text(np.mean(S2_BAND[p]), 1.02, "used", transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=5, color=INK2)
        if i == len(S2_ROWS) - 1:
            if p == "b":
                ax.set_xlabel("Weight of the window average", labelpad=3)
            if p == "c":                                          # bracket below the refitted heads
                tr = ax.get_xaxis_transform(); yb = -0.30
                ax.plot([0.7, 0.7, len(S2_HEADS) - 0.7, len(S2_HEADS) - 0.7], [yb + 0.04, yb, yb, yb + 0.04], color=AXIS, lw=0.5, transform=tr, clip_on=False, solid_capstyle="butt")
                ax.text((len(S2_HEADS)) / 2, yb - 0.035, "Refitted head, penalty weight γ", transform=tr, ha="center", va="top", fontsize=6, color=INK)
save_and_show(fig, "SuppFig2")

## Supplementary Fig. 3

**Supplementary Fig. 3 | Computational cost of the final setting compared with the baseline.** The baseline fine-tunes the model once and reads each segment once. The final setting fine-tunes the model twice, first as a teacher and then as a student, fits the penalized head, and uses mixed reading. **a**, Time of adaptation (3,000 steps for each fine-tuning run; the teacher bar includes the calculation of the teacher outputs). Adaptation is done once, before the model is used, and adds nothing to the time of reading. **b**, Time to read one segment. The final setting took two to four times as long, but at most 25 ms, which is negligible in absolute terms. **c**, Time to read 100,000 segments with a batch size of 512, calculated from the measured time for each segment. The final setting took 14–26 times as long, but at most 24 min, which remains practical when many ECGs are read at once. Values in parentheses are ratios to the baseline. Each training time was measured once.

Data: the two files that notebook 06 wrote to `TIMING` (their columns are documented in the last section of that notebook).

| Panel | File | Rows | Columns |
|---|---|---|---|
| **a** | `adaptation_time.csv` | baseline: `part` = `total`. Final setting: `teacher_run` + `teacher_outputs` (drawn together as the teacher run), `student_run`, `head_fitting`, and `total` for the numbers behind the bar | `minutes`, `ratio_to_baseline` |
| **b** | `reading_time.csv` | `task` = `one_segment` | `milliseconds_per_segment`, `ratio_to_baseline` |
| **c** | `reading_time.csv` | `task` = `many_segments` | `minutes_per_100000_segments`, `ratio_to_baseline`, `segments_per_batch` |
| line below the panels | both | `setting` = `final` | largest `peak_gpu_gb` of each file; `gpu` |

The ratios are read from the files and not calculated from the two bars. For one segment the ratio was measured within each round of the measurement,
so it can differ slightly from the quotient of the two times that are printed.

In [ ]:
for _f in ("adaptation_time.csv", "reading_time.csv"):
    if not (TIMING / _f).exists():
        raise FileNotFoundError(f"{_f} is missing in TIMING: run notebook 06 first")
ADAPTATION = pd.read_csv(TIMING / "adaptation_time.csv")
READING = pd.read_csv(TIMING / "reading_time.csv")
if ADAPTATION["quick_test"].astype(str).eq("True").any():
    raise ValueError("adaptation_time.csv contains times of a quick test: run notebook 06 with QUICK_TEST = False")
COST_SEGMENTS = 100_000       # panel c: number of segments; notebook 06 gives the time to read this many segments in the column minutes_per_100000_segments


def adaptation(column, setting, part):
    """One column of the adaptation times for one setting and one part: a value for each foundation model."""
    q = ADAPTATION[(ADAPTATION["setting"] == setting) & (ADAPTATION["part"] == part)].set_index("model")
    return q.loc[MODELS, column]


def reading(column, task, setting):
    """One column of the reading times for one task and one setting: a value for each foundation model."""
    q = READING[(READING["task"] == task) & (READING["setting"] == setting)].set_index("model")
    return q.loc[MODELS, column]


display(ADAPTATION[ADAPTATION["part"] == "total"].drop(columns=["part", "quick_test"]).round(2))
display(READING.drop(columns=["forward_passes"]).round(2))

### Drawing

Two bars for each foundation model: the baseline (grey) and the final setting. In **a** the bar of the final setting is divided into its three steps.
Behind each bar of the final setting are its time and, in parentheses, its ratio to the baseline (one decimal below 10).
The line below the panels names the graphics card and gives the largest peak memory of the final setting in adaptation and in reading.

In [ ]:
def draw_supp_fig3():
    """Computational cost from the two timing tables."""
    grey = "#b9b8b2"; teacher_color = tint(C_TRAIN, 0.45)
    n_train = int((pd.read_csv(PTBXL_SETS)["set"] == "train").sum())
    epochs = TRAIN_STEPS * BATCH / (2 * n_train)                  # a step uses BATCH labelled segments, and every training recording has two segments
    fig = figure(100)

    def two_bars(ax, baseline, final, ratio, unit, decimals, xmax):
        for i, m in enumerate(MODELS):
            y0, y1 = i - 0.19, i + 0.19
            ax.barh(y0, baseline[m], height=0.32, color=grey, zorder=3)
            ax.text(baseline[m] + xmax * 0.012, y0, f"{baseline[m]:.{decimals}f} {unit}", va="center", ha="left", fontsize=5.5, color=INK2)
            times = f"{ratio[m]:.1f}" if ratio[m] < 9.95 else f"{ratio[m]:.0f}"       # one decimal below 10
            ax.text(final[m] + xmax * 0.012, y1, f"{final[m]:.{decimals}f} {unit}   (×{times})", va="center", ha="left", fontsize=5.5, color=INK)
        ax.set_yticks(range(len(MODELS))); ax.set_yticklabels([MODEL_NAMES[m] for m in MODELS]); ax.set_ylim(len(MODELS) - 0.45, -0.55); ax.set_xlim(0, xmax); grid(ax, "x")

    # a: adaptation
    panel_letter(fig, 2, 2, "a"); text_mm(fig, 8, 4.2, "Adaptation", fontsize=7, fontweight="bold", va="center", ha="left")
    text_mm(fig, 8, 9.2, f"Length of a fine-tuning run: {TRAIN_STEPS:,} steps (batch size {BATCH}), about {epochs:.0f} epochs", fontsize=5.8, color=INK2, va="center", ha="left")
    ax = axes_mm(fig, 27, 15, 62, 62)
    two_bars(ax, adaptation("minutes", "baseline", "total"), adaptation("minutes", "final", "total"), adaptation("ratio_to_baseline", "final", "total"), "min", 0, 88)
    steps = [(adaptation("minutes", "final", "teacher_run") + adaptation("minutes", "final", "teacher_outputs"), teacher_color),       # the teacher run with the calculation of its outputs
             (adaptation("minutes", "final", "student_run"), C_TRAIN), (adaptation("minutes", "final", "head_fitting"), C_HEAD)]
    for i, m in enumerate(MODELS):
        left = 0
        for minutes, color in steps:
            ax.barh(i + 0.19, minutes[m], left=left, height=0.32, color=color, edgecolor="white", linewidth=0.5, zorder=3); left += minutes[m]
    ax.legend(handles=[Patch(color=grey, label="Baseline: fine-tuning run"), Patch(color=teacher_color, label="Final: teacher run"), Patch(color=C_TRAIN, label="Final: student run"), Patch(color=C_HEAD, label="Final: head fitting")],
              loc="upper right", handlelength=0.9, borderaxespad=0.3)
    ax.set_xlabel("Time (minutes)")
    # b: one segment
    panel_letter(fig, 99, 2, "b"); text_mm(fig, 105, 4.2, "Reading one ECG segment", fontsize=7, fontweight="bold", va="center", ha="left")
    text_mm(fig, 105, 9.2, f"Baseline: one read.  Final: one read and {N_WIN_READ} short windows, processed together", fontsize=5.8, color=INK2, va="center", ha="left")
    ax = axes_mm(fig, 124, 15, 50, 24)
    final = reading("milliseconds_per_segment", "one_segment", "final")
    two_bars(ax, reading("milliseconds_per_segment", "one_segment", "baseline"), final, reading("ratio_to_baseline", "one_segment", "final"), "ms", 0, 40)
    for i, m in enumerate(MODELS):
        ax.barh(i + 0.19, final[m], height=0.32, color=C_READ, zorder=3)
    ax.set_xlabel("Time (milliseconds; 1,000 ms = 1 s)")
    # c: many segments, read in batches
    column = f"minutes_per_{COST_SEGMENTS}_segments"
    batch_sizes = READING.loc[READING["task"] == "many_segments", "segments_per_batch"].unique()
    assert len(batch_sizes) == 1, "the rows many_segments of reading_time.csv have more than one batch size"
    panel_letter(fig, 99, 50, "c"); text_mm(fig, 105, 52.2, f"Reading {COST_SEGMENTS:,} ECG segments (batch size {int(batch_sizes[0])})", fontsize=7, fontweight="bold", va="center", ha="left")
    text_mm(fig, 105, 57.2, f"Baseline: one read for each segment.  Final: {1 + N_WIN_READ} reads for each segment", fontsize=5.8, color=INK2, va="center", ha="left")
    ax = axes_mm(fig, 124, 63, 50, 24)
    final = reading(column, "many_segments", "final")
    two_bars(ax, reading(column, "many_segments", "baseline"), final, reading("ratio_to_baseline", "many_segments", "final"), "min", 1, 36)
    for i, m in enumerate(MODELS):
        ax.barh(i + 0.19, final[m], height=0.32, color=C_READ, zorder=3)
    ax.set_xlabel("Time (minutes)")
    # line below the panels: graphics card and peak memory of the final setting
    cards = sorted(set(ADAPTATION["gpu"].dropna()) | set(READING["gpu"].dropna()))
    card = " / ".join(c.replace("GeForce ", "") for c in cards)   # the paper names the card without the product line: NVIDIA RTX 3080 (10 GB)
    memory_adaptation = ADAPTATION.loc[ADAPTATION["setting"] == "final", "peak_gpu_gb"].max(); memory_reading = READING.loc[READING["setting"] == "final", "peak_gpu_gb"].max()
    text_mm(fig, 8, 96.5, f"Graphics card: one {card}.   Peak memory used: at most {memory_adaptation:.1f} GB in adaptation and {memory_reading:.1f} GB in reading.",
            fontsize=5.8, color=INK2, va="center", ha="left")
    return fig


save_and_show(draw_supp_fig3(), "SuppFig3")

## Files written

The sixteen files in `FIGURES`: every figure as `.png` and as `.pdf`.

In [ ]:
for _name in [f"Fig{i}" for i in range(1, 6)] + [f"SuppFig{i}" for i in range(1, 4)]:
    for _ext in (".png", ".pdf"):
        _f = FIGURES / f"{_name}{_ext}"
        print(f"{_f.name:14s} {_f.stat().st_size / 1024:7.0f} kB")